# `examples/analysis` 结果分析示例 Notebook

本 Notebook 对应 `examples/analysis/main.py`，用于交互式运行 faas-sim 的基础仿真示例，并从 `Metrics` 中提取不同类型的指标表，便于观察一次仿真实验中发生了哪些部署、调度、调用、网络传输和资源使用事件。

## 包结构说明

```text
examples/analysis/
  __init__.py
  main.py
```

- `__init__.py`：将 `examples/analysis` 标记为 Python 包，便于通过模块方式导入。
- `main.py`：结果分析示例，负责构造仿真、运行仿真、读取指标并输出关键统计结果。

## 本 Notebook 的执行逻辑

1. 配置项目根目录和日志；
2. 导入 `examples.basic` 中的基础拓扑和基准测试；
3. 导入自定义函数仿真器工厂 `CustomSimulatorFactory`；
4. 创建并运行 `Simulation`；
5. 从 `sim.env.metrics` 中提取各类 DataFrame；
6. 对函数调用、调度、部署、网络流等结果进行快速查看。


In [55]:
# 标准库：用于定位项目根目录、处理模块导入路径、输出日志。
import sys
import logging
from pathlib import Path

# 第三方库：用于表格展示和统计分析。
import pandas as pd
from IPython.display import display

# -----------------------------------------------------------------------------
# 项目根目录定位
# -----------------------------------------------------------------------------
# 在 Jupyter 中运行示例时，当前工作目录可能是 notebooks、examples/analysis，
# 也可能是项目根目录。为了保证 `import sim`、`import examples` 均能正常工作，
# 这里从当前目录向上查找，直到找到同时包含 sim/ 和 examples/ 的目录。
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "sim").exists() and (PROJECT_ROOT / "examples").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

# 将项目根目录加入 Python 模块搜索路径最前面，确保优先使用当前项目中的 sim、ether、skippy、simpy 等内置包。
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"当前识别到的项目根目录：{PROJECT_ROOT}")

# -----------------------------------------------------------------------------
# 日志配置
# -----------------------------------------------------------------------------
# basicConfig 只在当前进程第一次配置 logging 时生效。
# level=logging.INFO 表示输出 INFO 及以上级别日志，便于观察仿真启动、运行和分析过程。
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")
logger = logging.getLogger("examples.analysis.notebook")


当前识别到的项目根目录：D:\研\7毕设\2 大论文\修辅\07-06\faas-sim-master


## 1. 导入仿真示例组件

这里复用 `examples/basic/main.py` 中已经定义好的基础拓扑和基准测试，并复用 `examples/custom_function_sim/main.py` 中的 `CustomSimulatorFactory`。这样可以让分析示例直接运行一个完整仿真，而不需要在 Notebook 中重新定义拓扑、函数部署和请求生成逻辑。


In [56]:
# examples.basic.main：提供基础示例中的拓扑构造函数 example_topology() 和 ExampleBenchmark。
# - example_topology()：创建一个可用于仿真的 Ether 网络拓扑。
# - ExampleBenchmark：定义仿真前的部署准备，以及仿真期间要生成的函数请求。
import examples.basic.main as basic

# CustomSimulatorFactory：自定义函数仿真器工厂。
# faas-sim 在部署函数副本时，会通过该工厂为每个 FunctionReplica 创建对应的 FunctionSimulator。
from examples.custom_function_sim.main import CustomSimulatorFactory

# Simulation：faas-sim 的仿真入口类，负责绑定拓扑、Benchmark 和 Environment，并驱动 SimPy 事件循环运行。
from sim.faassim import Simulation

logger.info("仿真依赖模块导入完成。")


INFO:examples.analysis.notebook:仿真依赖模块导入完成。


## 2. 创建并运行仿真

`Simulation` 接收两个核心输入：

- `topology`：由 Ether 定义的网络与节点拓扑；
- `benchmark`：由用户定义的实验场景，负责部署函数、注册镜像、生成请求等。

本示例中，我们创建基础拓扑和基础 Benchmark，然后将函数仿真器工厂替换为 `CustomSimulatorFactory`，最后调用 `sim.run()` 执行完整仿真。


In [57]:
def create_and_run_simulation():
    """
    创建并运行一次 faas-sim 仿真实验。

    业务作用：
    - 构造基础拓扑和基础 Benchmark；
    - 创建 Simulation 对象；
    - 指定函数副本使用的自定义 FunctionSimulator 工厂；
    - 启动仿真事件循环；
    - 返回运行完成后的 Simulation 对象，供后续提取 Metrics。

    返回：
    - sim: Simulation
      已运行完成的仿真实例，其中 sim.env.metrics 保存了仿真期间采集到的各类指标。
    """
    logger.info("开始创建基础拓扑。")
    topology = basic.example_topology()

    logger.info("开始创建基础 Benchmark。")
    benchmark = basic.ExampleBenchmark()

    logger.info("开始创建 Simulation 对象。")
    sim = Simulation(topology, benchmark)

    # 这里复用基础示例的拓扑和 benchmark，避免在分析示例中重复定义实验场景。
    # faas-sim 在部署函数副本时，需要为每个副本创建一个 FunctionSimulator。
    # 这里将工厂替换为 custom_function_sim 示例中的 CustomSimulatorFactory，
    # 用于演示“仿真系统 + 自定义函数执行模型 + 结果分析”的组合方式。
    sim.create_simulator_factory = CustomSimulatorFactory

    logger.info("开始运行仿真。")
    sim.run()
    logger.info("仿真运行完成。")

    return sim


sim = create_and_run_simulation()


INFO:examples.analysis.notebook:开始创建基础拓扑。
INFO:examples.analysis.notebook:开始创建基础 Benchmark。
INFO:examples.analysis.notebook:开始创建 Simulation 对象。
INFO:examples.analysis.notebook:开始运行仿真。
INFO:sim.faassim:initializing simulation, benchmark: ExampleBenchmark, topology nodes: 126
INFO:sim.faassim:starting resource monitor
INFO:sim.faassim:setting up benchmark
INFO:examples.basic.main:python-pi-cpu, latest, [ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='arm32'), ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='x86'), ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='aarch64')]
INFO:examples.basic.main:resnet50-inference-cpu, latest, [ImageProperties(name='resnet50-inference-cpu', size=56000000, tag='latest', arch='arm32'), ImageProperties(name='resnet50-inference-cpu', size=56000000, tag='latest', arch='x86'), ImageProperties(name='resnet50-inference-cpu', size=56000000, tag='latest', arch='aarch64')]
INFO:examples.basi

## 3. 提取 Metrics 指标表

faas-sim 的 `Metrics` 会在仿真过程中记录不同类型的事件。每一类事件可以通过 `extract_dataframe(name)` 转换为 Pandas DataFrame。

下面提取的指标名称与 `examples/analysis/main.py` 保持一致。


In [58]:
# 指标名称清单。
# key 是 Notebook 中使用的 DataFrame 名称，value 是 Metrics 内部记录的事件类型名称。
METRIC_TABLES = {
    "allocation_df": "allocation",                                       # 资源分配事件，例如函数副本占用节点资源。
    "invocations_df": "invocations",                                     # 函数调用事件，包括执行时间、开始/结束时间等。
    "scale_df": "scale",                                                 # 伸缩事件，例如 scale_up / scale_down。
    "schedule_df": "schedule",                                           # 调度事件，记录副本被调度到哪个节点及其调度耗时。
    "replica_deployment_df": "replica_deployment",                       # 函数副本部署事件，记录副本部署到节点的过程。
    "function_deployments_df": "function_deployments",                   # 函数部署集合信息。
    "function_deployment_df": "function_deployment",                     # 单个函数部署信息。
    "function_deployment_lifecycle_df": "function_deployment_lifecycle", # 函数部署生命周期事件。
    "functions_df": "functions",                                         # 函数元信息。
    "flow_df": "flow",                                                   # Ether 网络流事件，例如镜像拉取、数据传输产生的 flow。
    "network_df": "network",                                             # 网络层统计信息。
    "node_utilization_df": "node_utilization",                           # 节点资源利用率监控结果。
    "function_utilization_df": "function_utilization",                   # 函数副本资源利用率监控结果。
    "fets_df": "fets",                                                   # Function Execution Time 相关记录。
}


def extract_metric_dataframes(simulation):
    """
    从 Simulation 的 Metrics 中提取所有关心的指标表。

    参数：
    - simulation: Simulation
      已运行完成的仿真实例。

    返回：
    - dfs: dict[str, pandas.DataFrame]
      指标表字典，键为 Notebook 中使用的变量名，值为对应的 DataFrame。

    说明：
    - 有些指标只有在启用特定组件时才会产生数据；
    - 例如没有开启资源监控时，node_utilization_df 可能为空；
    - 没有触发网络传输时，flow_df / network_df 可能为空；
    - 因此这里用 try/except 做容错，方便先跑通样例再逐步扩展。
    """
    dfs = {}

    for df_name, metric_name in METRIC_TABLES.items():
        try:
            df = simulation.env.metrics.extract_dataframe(metric_name)
        except Exception as exc:
            logger.warning("提取指标表 %s 失败：%s", metric_name, exc)
            df = pd.DataFrame()

        dfs[df_name] = df
        logger.info("指标表 %-38s -> 行数：%s，列数：%s", df_name, len(df), len(df.columns))

    return dfs


dfs = extract_metric_dataframes(sim)


INFO:examples.analysis.notebook:指标表 allocation_df                          -> 行数：2，列数：3
INFO:examples.analysis.notebook:指标表 invocations_df                         -> 行数：20，列数：8
INFO:examples.analysis.notebook:指标表 scale_df                               -> 行数：2，列数：2
INFO:examples.analysis.notebook:指标表 schedule_df                            -> 行数：6，列数：6
INFO:examples.analysis.notebook:指标表 replica_deployment_df                  -> 行数：8，列数：4
INFO:examples.analysis.notebook:指标表 function_deployments_df                -> 行数：2，列数：2
INFO:examples.analysis.notebook:指标表 function_deployment_df                 -> 行数：2，列数：5
INFO:examples.analysis.notebook:指标表 function_deployment_lifecycle_df       -> 行数：2，列数：3
INFO:examples.analysis.notebook:指标表 functions_df                           -> 行数：0，列数：0
INFO:examples.analysis.notebook:指标表 flow_df                                -> 行数：2，列数：5
INFO:examples.analysis.notebook:指标表 network_df                             -> 行数：0，列数：0
INFO:examples.analysis.notebook

## 4. 查看各指标表规模

这一步先不深入分析具体业务结果，只查看每个 DataFrame 是否有数据、包含多少行和多少列。这样便于判断当前仿真是否产生了调用、调度、部署、网络或资源监控记录。


In [59]:
def metric_overview(dfs):
    """
    汇总每个指标表的规模信息。

    参数：
    - dfs: dict[str, pandas.DataFrame]
      extract_metric_dataframes 返回的指标表字典。

    返回：
    - overview_df: pandas.DataFrame
      每个指标表的行数、列数和字段名称摘要。
    """
    rows = []

    for name, df in dfs.items():
        rows.append({
            "指标表": name,
            "行数": len(df),
            "列数": len(df.columns),
            "字段": ", ".join(map(str, df.columns[:8])) + (" ..." if len(df.columns) > 8 else ""),
        })

    return pd.DataFrame(rows)


overview_df = metric_overview(dfs)
display(overview_df)


,指标表,行数,列数,字段
0,allocation_df,2,3,"cpu, mem, node"
1,invocations_df,20,8,"t_wait, t_exec, t_start, memory, function_name..."
2,scale_df,2,2,"value, function_name"
3,schedule_df,6,6,"value, function_name, image, replica_id, node_..."
4,replica_deployment_df,8,4,"value, function_name, node_name, replica_id"
5,function_deployments_df,2,2,"name, type"
6,function_deployment_df,2,5,"value, name, image, function_id, node"
7,function_deployment_lifecycle_df,2,3,"value, name, function_id"
8,functions_df,0,0,
9,flow_df,2,5,"bytes, duration, source, sink, action_type"


## 5. 分析函数调用执行时间

`invocations_df` 是最常用的指标表之一。它通常记录函数请求的执行时间、请求开始时间、目标函数、目标副本等信息。

原始 `main.py` 中的统计逻辑是：

```python
logger.info('Mean exec time %d', dfs['invocations_df']['t_exec'].mean())
```

下面在 Notebook 中做更稳健的版本：只有当 `invocations_df` 非空且存在 `t_exec` 字段时，才计算均值、最小值、最大值和分位数。


In [60]:
def analyze_invocations(invocations_df):
    """
    分析函数调用执行时间。

    参数：
    - invocations_df: pandas.DataFrame
      函数调用指标表，通常包含 t_exec 字段。

    返回：
    - stats_df: pandas.DataFrame
      执行时间统计结果。如果缺少数据，则返回空表。
    """
    if invocations_df.empty:
        logger.warning("invocations_df 为空，当前仿真没有记录函数调用事件。")
        return pd.DataFrame()

    if "t_exec" not in invocations_df.columns:
        logger.warning("invocations_df 中不存在 t_exec 字段，无法计算函数执行时间。当前字段：%s", list(invocations_df.columns))
        return pd.DataFrame()

    stats = {
        "调用次数": len(invocations_df),
        "平均执行时间": invocations_df["t_exec"].mean(),
        "最短执行时间": invocations_df["t_exec"].min(),
        "最长执行时间": invocations_df["t_exec"].max(),
        "P50执行时间": invocations_df["t_exec"].quantile(0.50),
        "P95执行时间": invocations_df["t_exec"].quantile(0.95),
        "P99执行时间": invocations_df["t_exec"].quantile(0.99),
    }

    logger.info("Mean exec time %.6f", stats["平均执行时间"])
    return pd.DataFrame([stats])


invocation_stats_df = analyze_invocations(dfs["invocations_df"])
display(invocation_stats_df)

# 展示前几条调用记录，便于观察字段含义。
if not dfs["invocations_df"].empty:
    display(dfs["invocations_df"].head())


INFO:examples.analysis.notebook:Mean exec time 1.250000


,调用次数,平均执行时间,最短执行时间,最长执行时间,P50执行时间,P95执行时间,P99执行时间
0,20,1.25,0.5,2.0,1.25,2.0,2.0


,t_wait,t_exec,t_start,memory,function_name,function_image,node,replica_id
time,,,,,,,,
2026-07-08 19:27:53.564433,0.0,0.5,11.0,1073741824,resnet50-inference,resnet50-inference-gpu,server_60,2672798589328
2026-07-08 19:27:53.564452,0.0,0.5,11.0,1073741824,resnet50-inference,resnet50-inference-gpu,server_60,2672798589328
2026-07-08 19:27:53.564463,0.0,0.5,11.0,1073741824,resnet50-inference,resnet50-inference-gpu,server_60,2672798589328
2026-07-08 19:27:53.564472,0.0,0.5,11.0,1073741824,resnet50-inference,resnet50-inference-gpu,server_60,2672798589328
2026-07-08 19:27:53.564480,0.0,0.5,11.0,1073741824,resnet50-inference,resnet50-inference-gpu,server_60,2672798589328


## 6. 查看调度与副本部署结果

调度与部署结果用于回答：

- 函数副本被调度到了哪些节点？
- 调度器是否记录了可行节点、目标节点或镜像拉取信息？
- 函数部署和副本部署是否按预期发生？

这些信息对后续理解 Skippy 调度器、节点资源约束、镜像本地性和冷启动过程很重要。


In [61]:
# 调度事件：观察调度器每次选择节点时记录了哪些信息。
if dfs["schedule_df"].empty:
    logger.warning("schedule_df 为空，当前仿真没有记录调度事件。")
else:
    print("schedule_df 前 5 行：")
    display(dfs["schedule_df"].head())

# 函数副本部署事件：观察每个副本部署过程。
if dfs["replica_deployment_df"].empty:
    logger.warning("replica_deployment_df 为空，当前仿真没有记录副本部署事件。")
else:
    print("replica_deployment_df 前 5 行：")
    display(dfs["replica_deployment_df"].head())

# 函数部署生命周期：观察函数部署从创建到可用的生命周期事件。
if dfs["function_deployment_lifecycle_df"].empty:
    logger.warning("function_deployment_lifecycle_df 为空，当前仿真没有记录函数部署生命周期事件。")
else:
    print("function_deployment_lifecycle_df 前 5 行：")
    display(dfs["function_deployment_lifecycle_df"].head())


schedule_df 前 5 行：


,value,function_name,image,replica_id,node_name,successful
time,,,,,,
2026-07-08 19:27:53.495893,queue,python-pi,python-pi-cpu,2672798586192,NaN,NaN
2026-07-08 19:27:53.496782,queue,resnet50-inference,resnet50-inference-gpu,2672798589328,NaN,NaN
2026-07-08 19:27:53.496836,start,python-pi,python-pi-cpu,2672798586192,NaN,NaN
2026-07-08 19:27:53.545945,finish,python-pi,python-pi-cpu,2672798586192,server_60,True
2026-07-08 19:27:53.547618,start,resnet50-inference,resnet50-inference-gpu,2672798589328,NaN,NaN


replica_deployment_df 前 5 行：


,value,function_name,node_name,replica_id
time,,,,
2026-07-08 19:27:53.547335,deploy,python-pi,server_60,2672798586192
2026-07-08 19:27:53.548723,deploy,resnet50-inference,server_60,2672798589328
2026-07-08 19:27:53.549314,startup,resnet50-inference,server_60,2672798589328
2026-07-08 19:27:53.550040,startup,python-pi,server_60,2672798586192
2026-07-08 19:27:53.551113,setup,resnet50-inference,server_60,2672798589328


function_deployment_lifecycle_df 前 5 行：


,value,name,function_id
time,,,
2026-07-08 19:27:53.495047,deploy,python-pi,2672798584400
2026-07-08 19:27:53.496011,deploy,resnet50-inference,2672798581488


## 7. 查看网络流与资源监控结果

网络流和资源监控并不一定在每个示例中都会产生完整数据。是否有数据取决于：

- 是否发生镜像拉取或数据传输；
- 是否启用了网络流记录；
- 是否启用了 `ResourceMonitor`；
- 函数仿真器是否在生命周期中记录资源申请与释放。


In [62]:
# 网络流事件：通常用于分析镜像拉取、数据传输、链路竞争等。
if dfs["flow_df"].empty:
    logger.warning("flow_df 为空，当前仿真没有记录网络流事件。")
else:
    print("flow_df 前 5 行：")
    display(dfs["flow_df"].head())

# 网络统计事件：用于分析网络层整体状态。
if dfs["network_df"].empty:
    logger.warning("network_df 为空，当前仿真没有记录网络统计事件。")
else:
    print("network_df 前 5 行：")
    display(dfs["network_df"].head())

# 节点资源利用率：需要 ResourceMonitor 或对应监控逻辑产生数据。
if dfs["node_utilization_df"].empty:
    logger.warning("node_utilization_df 为空，当前仿真没有记录节点资源利用率。")
else:
    print("node_utilization_df 前 5 行：")
    display(dfs["node_utilization_df"].head())

# 函数资源利用率：需要函数仿真器在执行阶段记录资源占用。
if dfs["function_utilization_df"].empty:
    logger.warning("function_utilization_df 为空，当前仿真没有记录函数资源利用率。")
else:
    print("function_utilization_df 前 5 行：")
    display(dfs["function_utilization_df"].head())


flow_df 前 5 行：


,bytes,duration,source,sink,action_type
time,,,,,
2026-07-08 19:27:53.549301,56000000,0.928571,registry,server_60,docker_pull
2026-07-08 19:27:53.550024,58000000,0.945004,registry,server_60,docker_pull


function_utilization_df 前 5 行：


,cpu,cpu_util,mem_util,node,replica_id
time,,,,,
2026-07-08 19:27:53.564607,88000.0,1.0,0,server_60,2672798586192
2026-07-08 19:27:53.564629,0.0,0.0,0,server_60,2672798589328
2026-07-08 19:27:53.564740,0.0,0.0,0,server_60,2672798586192
2026-07-08 19:27:53.564754,0.0,0.0,0,server_60,2672798589328


## 8. 按需导出分析结果

如果希望后续用 Excel、Origin、Python 脚本或论文绘图程序继续处理结果，可以将 DataFrame 导出为 CSV。默认下面代码不执行批量导出，取消注释后即可使用。


In [63]:
# 输出目录：默认放在项目根目录下的 outputs/examples_analysis。
# 取消注释后会创建目录，并把所有非空指标表导出为 CSV 文件。

# output_dir = PROJECT_ROOT / "outputs" / "examples_analysis"
# output_dir.mkdir(parents=True, exist_ok=True)
#
# for name, df in dfs.items():
#     if not df.empty:
#         output_path = output_dir / f"{name}.csv"
#         df.to_csv(output_path, index=False, encoding="utf-8-sig")
#         print(f"已导出：{output_path}")


## 9. 小结

这个 Notebook 的作用不是实现新的调度器或新的函数仿真器，而是作为结果分析入口，帮助你确认一次仿真实验是否产生了预期指标。

后续调试样例时，可以优先观察：

- `invocations_df`：函数调用是否发生，执行时间是否合理；
- `schedule_df`：调度是否发生，目标节点是否符合预期；
- `replica_deployment_df`：副本是否部署成功；
- `flow_df`：是否发生网络流，例如镜像拉取或数据传输；
- `node_utilization_df` / `function_utilization_df`：资源监控是否开启并产生数据。
